# Практическая работа № 3 — Пайплайн предобработки (уровень B)

**Студент:** Barsbek Konurbaev

Цель: построить воспроизводимый pipeline с отдельной обработкой числовых и категориальных признаков и без утечки данных.

## 1. Загрузка реального датасета
Используется UCI Adult Census Income. Загрузка требует интернет. Значения `?` трактуются как пропуски. Целевая переменная — `income`.

In [ ]:
adult_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = [
    "age", "workclass", "fnlwgt", "education", "education_num", "marital_status",
    "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
    "hours_per_week", "native_country", "income"
]
df = pd.read_csv(
    adult_url, header=None, names=columns, na_values="?",
    skipinitialspace=True, comment="|"
)
df = df.dropna(how="all").reset_index(drop=True)
print("Размер:", df.shape)
display(df.head())

## 2. Разделение признаков и целевой переменной
Сначала делим данные, затем обучаем все шаги предобработки только на train — это предотвращает утечку.

In [ ]:
X = df.drop(columns="income")
y = df["income"].str.strip().str.replace(".", "", regex=False)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

print("Числовые признаки:", numeric_features)
print("Категориальные признаки:", categorical_features)
print("Train:", X_train.shape, "Test:", X_test.shape)

## 3. ColumnTransformer и Pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("F1:", round(f1_score(y_test, pred, pos_label=">50K"), 4))
print(classification_report(y_test, pred, zero_division=0))

## 4. Проверка размерностей и отсутствия утечки

In [ ]:
X_train_transformed = model.named_steps["preprocessor"].transform(X_train)
X_test_transformed = model.named_steps["preprocessor"].transform(X_test)
print("Матрица train:", X_train_transformed.shape)
print("Матрица test:", X_test_transformed.shape)
print("Число признаков совпадает:", X_train_transformed.shape[1] == X_test_transformed.shape[1])
print("Импьютеры и кодировщик обучены только через model.fit(X_train, y_train).")

## 5. Обоснование стратегий
- Числовые пропуски заполняются медианой, устойчивой к выбросам.
- Категориальные пропуски заполняются наиболее частой категорией.
- Категориальные значения кодируются OneHotEncoder; неизвестные тестовые категории не вызывают ошибку.
- Числовые признаки масштабируются StandardScaler.
- Все шаги объединены в Pipeline и обучаются только на train.

## Вывод
Укажите метрики после запуска и подтвердите, что размерность train/test после преобразования совпадает.